# Дообучение нейронки EyePhoto

Перед запуском справа в панели **Settings**: *Accelerator* — **GPU T4 x2** или **GPU P100**, *Internet* — **On**. Слева **+ Add Input** → ваш датасет с глазами.

Запускать через **Save Version → Save & Run All (Commit)**: тогда обучение идёт на серверах Kaggle, даже если закрыть браузер. Результат появится во вкладке **Output** этой версии.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# скрипты обучения из репозитория программы
import os, urllib.request
BASE = 'https://raw.githubusercontent.com/chsts0/eyephoto/main/training/'
os.makedirs('/kaggle/working/training', exist_ok=True)
for f in ['train.py', 'lama_model.py', 'safe_ckpt.py']:
    urllib.request.urlretrieve(BASE + f, '/kaggle/working/training/' + f)
print(os.listdir('/kaggle/working/training'))

In [ ]:
# где лежат глаза: первый подключённый датасет
import glob
DATA = sorted(glob.glob('/kaggle/input/*'))[0]
n = len(glob.glob(DATA + '/**/eye_*_mask.png', recursive=True))
print(DATA, '— глаз:', n)
assert n > 0, 'Не нашёл глаз: подключите датасет от сборщика через + Add Input'

In [ ]:
# обучение: ~9 часов (Kaggle даёт до 12 за сессию). Если не хватает памяти, поменяйте --batch 4 на --batch 2
!python /kaggle/working/training/train.py --data "$DATA" --out /kaggle/working/out --hours 9 --batch 4 --cache /kaggle/working/cache

In [ ]:
# итог: график ошибки и последние сравнения (слева направо: вход, старая модель, новая, как было)
import pandas as pd, glob
from IPython.display import Image, display
print(pd.read_csv('/kaggle/working/out/log.csv'))
for p in sorted(glob.glob('/kaggle/working/out/samples/*.jpg'))[-2:]:
    print(p); display(Image(p, width=1000))

In [ ]:
# чтобы во вкладке Output не лежали лишние 1,5 ГБ: оставляем модель, примеры и лог
import shutil, os
shutil.rmtree('/kaggle/working/cache', ignore_errors=True)
for f in ['last.pt']:
    p = '/kaggle/working/out/' + f
    if os.path.exists(p): os.remove(p)
print(os.listdir('/kaggle/working/out'))